# Introduction to Preference Alignment with LLM-as-a-Judge (DPO)

This reference implements the **Con-J framework** from:

> - Ye, Z., et al., (2025).
Learning LLM-as-a-Judge for Preference Alignment.
https://openreview.net/forum?id=HZVIQE1MsJ

Con-J trains an LLM to act as a **generative judge** using Direct Preference Optimization (DPO), instead of relying on a scalar reward model.

## Why Not a Scalar Reward Model?

Traditional preference alignment (e.g., RLHF-style reward models) trains a model to assign a **scalar score** to each answer.

However, as shown in the following figure:

- Scalar models output only a number → ❌ No explanation  
- They are more susceptible to dataset bias  
- They may learn superficial patterns (e.g., verbosity)

Con-J instead trains an **LLM-as-a-Judge** that generates:

- A natural language **rationale**
- A binary **preference decision**

This improves:

- Interpretability  
- Robustness to bias  
- Alignment transparency 

![Figure 1: Scalar Reward Model vs Generative Judge](assets/Figure1.png)


## Con-J Training Pipeline

The full framework is illustrated in the following figure.

The process consists of three stages:

### 1️⃣ Judgment Sampling(Repeated and Hint Sampling)
Given a prompt `q` and answers `(a₁, a₂)`,  
the pretrained LLM generates multiple **judgments with rationales**.

### 2️⃣ Judgment Filtering
Using ground-truth preference labels, judgments are separated into:

- **Positive judgments** (correct preference)
- **Negative judgments** (incorrect or unclear preference)

These are paired to form **contrastive judgment pairs**.

### 3️⃣ Contrastive Training (DPO)
The LLM is trained using:

- **DPO loss** on contrastive pairs  
- A small **SFT loss** on positive judgments  

This directly optimizes the model to prefer correct judgments while maintaining generation quality.

![Figure 2: Con-J Pipeline](assets/Figure2.png)

## What This Notebook Does

Before DPO training, we must first construct the dataset required for contrastive learning.

This notebook performs:

- Conversion of `(prompt, chosen, rejected)` data into judge prompts  
- Generation of multiple LLM judgments  
- Filtering into positive/negative sets  
- Construction of contrastive pairs for DPO  

## Overall Workflow

1. Dataset Construction  
2. LLM-as-a-Judge Inference  
3. Contrastive Pair Construction  
4. DPO Training  
5. Evaluation  

This implementation mirrors the methodology proposed in the Con-J paper while keeping the pipeline modular and reproducible.


# Dataset Construction for Preference Alignment (DPO)

This notebook prepares the **judge prompts** used for LLM-as-a-Judge inference.

This notebook constructs preference-style datasets required for Direct Preference Optimization (DPO).

Given an existing dataset containing chosen and rejected responses, we convert each example into a judge-ready prompt that allows an LLM to compare two candidate answers and select the better one.

This follows the standard preference learning paradigm used in RLHF and DPO pipelines (Rafailov et al., 2023).


In [ ]:
from pathlib import Path
from utils.dataset_helpers import (
    set_seed,
    load_parquet_dataset,
    build_judge_dataset,
    save_dataset,
    preview_samples,
)

# Resolve Repository Root
try:
    REPO_ROOT = Path(__file__).resolve().parents[1]
except NameError:
    REPO_ROOT = Path.cwd()

# Select Dataset Folder
# Choose one of:
#   "data_sky"
#   "data_hh_rlhf"
DATA_FOLDER_NAME = "data_sky"  # <-- change as needed


DATA_DIR = REPO_ROOT / DATA_FOLDER_NAME

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"{DATA_DIR} does not exist. Please download or create the dataset folder."
    )

# Automatically Detect Parquet File
parquet_files = list(DATA_DIR.glob("*.parquet"))

if not parquet_files:
    raise FileNotFoundError(
        f"No .parquet files found in {DATA_DIR}. "
        "Please download or generate the filtered dataset and place it in this folder."
    )

PARQUET_PATH = parquet_files[0]

print(f"Using dataset file: {PARQUET_PATH}")

# Automatically Set Save Directory
# Example:
# data_sky        → output_sky
# data_hh_rlhf    → output_hh_rlhf

SAVE_DIR = REPO_ROOT / f"output_{DATA_FOLDER_NAME}"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Saving processed dataset to: {SAVE_DIR}")

# Automatically infer dataset format
if "hh" in DATA_FOLDER_NAME.lower():
    DATASET_FORMAT = "hh"
elif "sky" in DATA_FOLDER_NAME.lower():
    DATASET_FORMAT = "sky"
else:
    raise ValueError(
        "Unable to infer dataset format. "
        "Folder name must contain 'sky' or 'hh'."
    )

print(f"Detected dataset format: {DATASET_FORMAT}")


### Reproducibility Control

Since answer ordering is randomly flipped during dataset construction, we fix the random seed to ensure deterministic generation of preference pairs.

Reproducibility is critical in alignment research to ensure that observed improvements arise from model updates rather than stochastic sampling artifacts.

In [ ]:
set_seed(2021)

### Loading Source Dataset

We load the source dataset in Parquet format. Each record contains a prompt and two responses: one preferred (chosen) and one non-preferred (rejected).

This format aligns with common reward-model and DPO datasets used in alignment research.

In [ ]:
raw_dataset = load_parquet_dataset(str(PARQUET_PATH))
print(raw_dataset)

### Judge Prompt Construction

To enable LLM-based comparison, we transform each example into a structured evaluation prompt.

For each question, we present two candidate answers and instruct the judge model to select the better one based on:

- Coherence
- Accuracy
- Coverage
- Overall quality

We randomly flip answer ordering to prevent positional bias and ensure robustness of preference supervision.

This LLM-as-a-judge approach follows recent evaluation paradigms:

- Zheng et al., 2023 — Judging LLMs by Their Covers (MT-Bench)

- Liu et al., 2023 — G-Eval: NLG Evaluation using GPT-4

In [ ]:
train_dataset = build_judge_dataset(
    raw_dataset,
    dataset_format=DATASET_FORMAT
)
print(train_dataset)

In [ ]:
preview_samples(train_dataset, n=3)

### Persisting Dataset

The constructed dataset is saved in HuggingFace Dataset format to ensure compatibility with downstream DPO training and evaluation pipelines.

This structured storage enables reproducible experimentation and efficient loading during training.

In [ ]:
final_dataset = save_dataset(train_dataset, SAVE_DIR)
print("Dataset saved to:", SAVE_DIR)


References

- Rafailov et al., 2023
Direct Preference Optimization: Your Language Model is Secretly a Reward Model
https://arxiv.org/abs/2305.18290

- Zheng et al., 2023
Judging LLMs by Their Covers: A Benchmark for Open-ended Question Answering (MT-Bench & Chatbot Arena)
https://arxiv.org/abs/2306.05685